This is the **historical v5** training notebook, kept for comparison. The v7 submission notebook is `tally_sft.ipynb`.

# Tally — AlignTune LoRA fine-tune

This notebook fine-tunes **Tiny Aya** (`CohereLabs/tiny-aya-global`) so it gets better at extracting the seven Tally fields from a forest-carbon document chunk, and better at saying `"not stated"` when the chunk does not support a value.

It is a **new** notebook (`renhuang01/tally-sft`). It does not change the existing `tally-kaggle` extraction notebook.

**What it does, in order**

1. Installs AlignTune (Lexsi's fine-tuning library) and logs in to Hugging Face with your `HF_TOKEN` secret. The token is never printed.
2. Loads the curated training set (v5: **158 examples** from 20 projects, see below) and the held-out validation set (23 examples from Miro and Bandai Hills).
3. Runs a LoRA / QLoRA supervised fine-tune with AlignTune's TRL backend. If AlignTune cannot start on this machine, a clearly marked TRL + PEFT fallback does the same job.
4. Saves the adapter to `/kaggle/working/tally-lora`.
5. Re-runs extraction on the validation chunks with the fine-tuned model and with the base model, merges the chunk notes the same way as the extraction notebook, and scores both against the answer key using the same rules as `tinyaya_scores.json`.

**Before you press Save & Run All**

1. Settings → Accelerator → **GPU T4 x1** (T4 x2 is also fine; this notebook uses the first card only).
2. Settings → Internet → **on** (needed to install packages and download the model).
3. **Add data** → attach the private dataset `renhuang01/tally-sft-v5` and the private token dataset `renhuang01/tally-hf-token`.
4. Add-ons → Secrets → attach **`HF_TOKEN`** (the same secret the extraction notebook uses, so the gated Tiny Aya weights can be downloaded).
5. Then **Save & Run All**. Expect about 40–70 minutes: roughly 20–45 minutes of training and 15–25 minutes to rescore both models on the 23 validation chunks.

The model is about 7 GB in float16, so we train with **4-bit QLoRA**, micro-batch 1, an 8,704-token cap (long enough that no example loses its answer), gradient checkpointing, and gradient accumulation of 8. That should fit in the 16 GB of one T4. Outputs stay under `/kaggle/working`, which Kaggle keeps between sessions.


**v5 — more projects, two epochs (13 Nov 2026).** v4 (62 examples from 7 projects, 3 epochs) overfit: train loss fell from 0.54 to ~0.20 while val loss stayed at ~0.98, and stated accuracy on Miro and Bandai dropped from 3/13 to 0/13. This version trains on the v5 set: the 62 v4 examples plus 96 new chunks from 13 further African forest-carbon PDs, for **158 examples from 20 projects** (130 with at least one value, 28 all-`not stated`). Both `train_sft_messages.jsonl` and `train_sft_messages_rebalanced.jsonl` hold the same 158 rows. Validation is unchanged (the same 23 Miro + Bandai chunks). **2 epochs**, warmup about 6 steps (warmup_ratio 0.1). Everything else matches v4: fp16 QLoRA, LoRA r=16 / α=32 on `q_proj`, `k_proj`, `v_proj`, `o_proj`, loss only on the answer span.


## 1. Setup

Kaggle already has PyTorch. We install AlignTune and `bitsandbytes` (for 4-bit QLoRA), then log in to Hugging Face. Tiny Aya is a gated model, so the token is required. It is read from the `HF_TOKEN` secret, or, if that secret is not attached, from `hf_token.txt` in the private `renhuang01/tally-hf-token` dataset. It is never printed.

The weights are cached under `/kaggle/tmp/hf`, which is session scratch. We do not put the Hugging Face cache on `/kaggle/working`, because that cache uses shortcuts (symlinks) that have broken on some host filesystems before.


In [ ]:
# === 1. SETUP: install packages, log in, and set paths ===
# Run this after any session restart. Does not load the model yet;
# AlignTune / the TRL fallback will load it for training.

import os
from pathlib import Path

# Kaggle sometimes gives two T4s. Pin to one so the HF Trainer does not wrap the
# 4-bit model in DataParallel. Must run before torch is imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Less fragmentation on long, variable-length examples.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.environ["HF_HOME"] = "/kaggle/tmp/hf"
os.environ["HF_HUB_CACHE"] = "/kaggle/tmp/hf"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/tmp/hf"
os.makedirs("/kaggle/tmp/hf", exist_ok=True)
os.makedirs("/kaggle/working/tally", exist_ok=True)

# Do not reinstall torch. transformers>=5 matches the extraction notebook.
%pip install -q aligntune bitsandbytes "transformers>=5"

from huggingface_hub import login


def get_hf_token():
    # 1) the notebook's HF_TOKEN Kaggle secret; 2) a private attached dataset
    # file hf_token.txt (renhuang01/tally-hf-token). The token is never printed.
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok and tok.strip():
            return tok.strip(), "Kaggle secret"
    except Exception as exc:
        print("HF_TOKEN secret not available:", type(exc).__name__)
    for f in sorted(Path("/kaggle/input").rglob("hf_token.txt")):
        tok = f.read_text(encoding="utf-8").strip()
        if tok:
            return tok, "private dataset file"
    raise RuntimeError("No Hugging Face token: attach the HF_TOKEN secret or the private tally-hf-token dataset.")


_hf_tok, _hf_src = get_hf_token()
login(_hf_tok)
os.environ["HF_TOKEN"] = _hf_tok
print("Logged in to Hugging Face via", _hf_src)
del _hf_tok

import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}  "
              f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: no GPU visible. Turn on Settings → Accelerator → GPU T4 x1 and restart.")

print("cuda:", torch.cuda.is_available(), "torch", torch.__version__)
import importlib.metadata as _md
for _p in ("transformers", "trl", "peft", "datasets", "bitsandbytes", "accelerate", "aligntune", "curatorkit"):
    try:
        print(f"{_p}=={_md.version(_p)}")
    except Exception:
        print(_p, "not installed")


## 2. Find the curated dataset and inspect it

The private dataset `renhuang01/tally-sft-v5` should appear under `/kaggle/input/tally-sft-v5/`. Each training row is a chat example: a user message (our extraction prompt plus one document chunk) and an assistant message (the JSON we want Tiny Aya to produce, taken from the answer key). Chunks where a field is not supported are labelled `"not stated"`, so the model also practises abstaining.

The split is by **project**, not by chunk. Training is 158 examples from 20 projects (the original v4 projects plus 13 new Verra PDs). Validation is unchanged: 23 chunks from Miro and Bandai Hills. Held-out documents stay out of training, so a figure from the same report cannot leak into both sides.

A few source quotes contain the Unicode line separator U+2028, so we read JSONL by splitting only on `\n`, not with `str.splitlines()`.


In [ ]:
# === 2. PATHS: locate the attached dataset and load the JSONL files ===

import json
import os
import sys
from pathlib import Path

MID = "CohereLabs/tiny-aya-global"
HF_CACHE = "/kaggle/tmp/hf"
ADAPTER_DIR = Path("/kaggle/working/tally-lora")
RUN_DIR = Path("/kaggle/working/tally/finetune_runs")
RESULTS_DIR = Path("/kaggle/working/tally/finetune_results")
SFT_WORK = Path("/kaggle/working/sft_data")
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SFT_WORK.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("/kaggle/input/tally-sft-v5"),
    Path("/kaggle/input/tally-sft-curated"),
    *([Path(os.environ["TALLY_ROOT"]).expanduser() / rel for rel in ("sft_v5/dataset", "kaggle_sft_dataset", "data/curated")] if os.environ.get("TALLY_ROOT") else []),
    Path.cwd() / "data" / "curated",
]

def _has_train(p):
    return (p / "train_sft_messages.jsonl").is_file()


def _unzip_payload(zip_path):
    import zipfile
    dest = Path("/kaggle/working/tally_sft_input")
    dest.mkdir(parents=True, exist_ok=True)
    if not _has_train(dest):
        print("Unzipping", zip_path, "->", dest)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(dest)
    return dest


def find_data_dir():
    # Search every attached input recursively (Kaggle may mount datasets in nested folders)
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
    if os.environ.get("TALLY_ROOT"):
        roots.append(Path(os.environ["TALLY_ROOT"]).expanduser())
    for root in roots:
        if not root.exists():
            continue
        hits = sorted(root.rglob("train_sft_messages.jsonl"))
        if hits:
            print("Found data at", hits[0].parent)
            return hits[0].parent
        zips = sorted(root.rglob("tally_sft_files.zip"))
        if zips:
            return _unzip_payload(zips[0])
    import subprocess
    print(subprocess.run(["find", "/kaggle/input", "-maxdepth", "5"], capture_output=True, text=True).stdout)
    raise FileNotFoundError("train_sft_messages.jsonl not found under /kaggle/input; see listing above.")

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

DATA_DIR = find_data_dir()
sys.path.insert(0, str(DATA_DIR))

# v5: both train_sft_messages.jsonl and train_sft_messages_rebalanced.jsonl
# are the same 158-row train set (62 v4 examples + 96 new chunks from 13 PDs).
# Prefer the rebalanced filename when present (same bytes on v5); fall back to
# train_sft_messages.jsonl. Validation is unchanged (same 23 Miro + Bandai
# chunks), so scores stay comparable with v4.
REBALANCED = DATA_DIR / "train_sft_messages_rebalanced.jsonl"
if REBALANCED.is_file():
    train_rows = load_jsonl(REBALANCED)
    print("Using train split:", REBALANCED.name)
    _rep = DATA_DIR / "rebalance_report.json"
    if _rep.is_file():
        _r = json.loads(_rep.read_text(encoding="utf-8"))
        print({k: _r[k] for k in ("seed", "n_pos_examples", "n_neg_sampled", "n_additionality_dups", "n_train", "field_positive_counts", "max_tokens_after_trim") if k in _r})
else:
    train_rows = load_jsonl(DATA_DIR / "train_sft_messages.jsonl")
    print("Using train split: train_sft_messages.jsonl")
_n_empty = sum(1 for r in train_rows if all(v == "not stated" for v in json.loads(r["messages"][1]["content"]).values()))
print(f"train: {len(train_rows)} examples, {_n_empty} all-'not stated' ({_n_empty / len(train_rows):.0%})")
val_rows = load_jsonl(DATA_DIR / "val_sft_messages.jsonl")
val_meta = load_jsonl(DATA_DIR / "val_sft_with_metadata.jsonl")
answer_key = json.loads((DATA_DIR / "answer_key.json").read_text(encoding="utf-8"))
base_scores_all = json.loads((DATA_DIR / "tinyaya_scores.json").read_text(encoding="utf-8"))

from tally_merge import FIELDS, merge_notes
from score_extractions import (
    score_documents,
    filter_scores_to_docs,
    format_summary_table,
)

print("DATA_DIR =", DATA_DIR)
print("train examples:", len(train_rows), "  val examples:", len(val_rows))
print("val documents:", sorted({r['doc'] for r in val_meta}))

# Show one shortened training example so you can see the format.
ex = train_rows[0]
print("\nExample 0 roles:", [m['role'] for m in ex['messages']])
user = ex['messages'][0]['content']
asst = ex['messages'][1]['content']
print("user (first 600 chars):")
print(user[:600], "...")
print("\nassistant target:")
print(asst)

# Copy train/val into a local folder AlignTune / TRL can read.
# (Kaggle input is read-only.)
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

write_jsonl(SFT_WORK / "train.jsonl", train_rows)
write_jsonl(SFT_WORK / "validation.jsonl", val_rows)
print("\nwrote", SFT_WORK / "train.jsonl", "and", SFT_WORK / "validation.jsonl")


## 3. The training choices, in plain words

- **LoRA / QLoRA, not a full-weight update.** Tiny Aya has 3.35 billion parameters. Updating all of them, plus optimiser states, would not fit next to a long (up to 6,144-token) context on a 16 GB T4. LoRA trains a small set of extra weights on top of the frozen model. QLoRA also stores the frozen model in 4-bit, which drops the base weights from about 7 GB to about 2–2.5 GB and leaves room for the long context.
- **Loss only on the answer.** We do not want the model to memorise the prompt or the document chunk. Tiny Aya's chat template has no `{% generation %}` markers, so TRL's built-in `assistant_only_loss` cannot work with it. Instead the training loss is masked by hand to the tokens after `<|START_RESPONSE|>`: the JSON answer and its end markers. The same masked loss is used by AlignTune's TRL backend and the fallback. It also only computes the model's output scores (logits) for that answer span, which is what keeps an 8,000-token example inside 16 GB.
- **Micro-batch 1, accumulation 8.** One long example at a time (max 8,704 tokens), with eight of those added up before each optimiser step, for an effective batch size of 8.
- **Two epochs (v5; v4 used three), learning rate 2e-4, LoRA rank 16 on the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`).** v5 trains on 158 examples from 20 projects. Two passes give about 40 optimiser steps (158 / 8 accumulation × 2), with a short warmup of about 6 steps (or `warmup_ratio` 0.1). We log validation loss each epoch and keep the checkpoint with the best validation loss.
- **Backend is TRL, not Unsloth.** Unsloth is faster when it installs cleanly, but it is brittle on Kaggle. AlignTune's TRL backend is the documented default and is what we ask it to use.

These values come from AlignTune's public API (`create_sft_trainer` and the [parameter reference](https://aligntune.lexsi.ai/PARAMETERS/)), not from a guessed interface.


## 4. Train

The next cell tries **AlignTune first**. That is a Lexsi hackathon requirement, and CuratorKIT already exported this data in a format AlignTune accepts (`sft_messages`).

If AlignTune cannot be imported, or `create_sft_trainer` fails (for example a dependency clash on this Kaggle image), the same cell falls back to **TRL `SFTTrainer` + PEFT LoRA** with the same hyperparameters, and prints a clear note that the fallback was used. The saved adapter is the same either way. If both fail, the notebook keeps going and still scores the base model, so the run never dies halfway.


In [ ]:
# === 4. TRAIN: AlignTune QLoRA SFT, with a TRL+PEFT fallback ===
# This is the GPU-heavy cell. It may take 20-45 minutes.
# Neither backend is allowed to crash the notebook: if both fail, the
# rescoring cells still run the base model and report the failure.

import gc
import inspect
import shutil
import traceback

import torch
import torch.nn.functional as F
from datasets import Dataset

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Hyperparameters (shared by AlignTune and the fallback).
NUM_EPOCHS = 2   # v5: 158 examples from 20 projects; two passes to limit overfitting
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
# Longest training example is ~8,440 tokens with the chat template. A cap
# below that would cut off the JSON answer (it sits at the end), so we keep
# every example whole. Memory stays low because the loss only materialises
# logits for the answer span (see tally_compute_loss).
MAX_SEQ_LENGTH = 8704
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
# Cohere2 (Tiny Aya) attention projections, verified in
# transformers/models/cohere2/modeling_cohere2.py: q_proj, k_proj, v_proj, o_proj.
# (MLP: gate_proj, up_proj, down_proj.) Same set AlignTune picked in run 2.
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]
SEED = 42
# ~6 warmup steps: 158 examples x 2 epochs / 8 accumulation = ~40 optimiser steps
# (warmup_ratio 0.1 is ~4 steps; WARMUP_STEPS=6 is used by the TRL fallback)
WARMUP_STEPS = 6

QUANTIZATION = {
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,
}

USED_BACKEND = None  # "aligntune-trl", "trl-peft-fallback", or None if both failed
TRAIN_METRICS = {}
TRAIN_ERRORS = {}
trainer = None

# ---- Answer-only loss --------------------------------------------------
# Tiny Aya's chat template has no {% generation %} tags, so TRL's
# assistant_only_loss cannot build a mask from it. Instead we mask by hand:
# only the tokens after the last <|START_RESPONSE|> (the JSON answer,
# <|END_RESPONSE|> and end-of-turn) count towards the loss. We also ask the
# model for logits on that span only (logits_to_keep), which avoids a
# ~8,000 x 262,144 logit tensor that would not fit on a T4.
_tok_for_ids = AutoTokenizer.from_pretrained(MID, cache_dir=HF_CACHE)
START_RESPONSE_ID = _tok_for_ids.convert_tokens_to_ids("<|START_RESPONSE|>")
assert START_RESPONSE_ID is not None and START_RESPONSE_ID != _tok_for_ids.unk_token_id, "START_RESPONSE token missing"
print("START_RESPONSE id:", START_RESPONSE_ID)


def tally_compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
    # Our loss is a per-token mean; let the Trainer divide by grad-accum steps.
    self.loss_is_scaled_for_ga = False
    self.model_accepts_loss_kwargs = False
    inputs.pop("_prediction_loss_only", None)
    ids = inputs["input_ids"]
    attn = inputs.get("attention_mask")
    if attn is None:
        attn = torch.ones_like(ids)
    B, L = ids.shape
    pos = torch.arange(L, device=ids.device).unsqueeze(0).expand(B, L)
    is_start = (ids == START_RESPONSE_ID) & (attn > 0)
    last_start = torch.where(is_start, pos, torch.full_like(pos, -1)).max(dim=1).values  # (B,)
    has_ans = last_start >= 0
    if not bool(has_ans.any()):
        # Answer was truncated away; contribute zero loss but keep the graph.
        zero = sum(p.sum() for p in model.parameters() if p.requires_grad) * 0.0
        return (zero, {"loss": zero}) if return_outputs else zero
    first_ans = torch.where(has_ans, last_start + 1, torch.full_like(last_start, L))
    K = int(L - int(first_ans[has_ans].min().item()) + 1)
    K = max(2, min(K, L))
    fwd = {"input_ids": ids, "attention_mask": attn, "use_cache": False, "logits_to_keep": K}
    if "position_ids" in inputs:
        fwd["position_ids"] = inputs["position_ids"]
    out = model(**fwd)
    logits = out.logits[:, :-1, :].float()          # predicts tokens L-K+1 .. L-1
    tgt = ids[:, L - K + 1:]
    tpos = pos[:, L - K + 1:]
    mask = (tpos >= first_ans.unsqueeze(1)) & (attn[:, L - K + 1:] > 0)
    ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), reduction="none").view_as(tgt)
    loss = (ce * mask).sum() / mask.sum().clamp(min=1)
    if not getattr(self, "_tally_logged", False):
        self._tally_logged = True
        print(f"[answer-only loss] seq len {L}, answer tokens {int(mask.sum())}, logits kept {K}")
    return (loss, {"loss": loss, "logits": logits[:, -1:, :]}) if return_outputs else loss


import trl
from trl import SFTTrainer


def cast_trainable_fp32(model):
    # fp16 AMP needs fp32 master weights for the trainable LoRA params.
    # (TRL's SFTTrainer.__init__ casts QLoRA adapters to bfloat16, which the
    # T4's fp16 GradScaler cannot unscale, so we undo that right before training.)
    n = 0
    for p in model.parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
            n += 1
    return n


_ORIG_TRAIN = SFTTrainer.train
_ORIG_TRAINING_STEP = SFTTrainer.training_step
OOM_SKIPS = []


def tally_train(self, *args, **kwargs):
    n = cast_trainable_fp32(self.model)
    dts = sorted({str(p.dtype) for p in self.model.parameters() if p.requires_grad})
    print(f"[pre-train] cast {n} trainable tensors to fp32; trainable dtypes now {dts}; "
          f"fp16={self.args.fp16} bf16={self.args.bf16} grad_ckpt={self.args.gradient_checkpointing}")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        print(f"[pre-train] GPU allocated {torch.cuda.memory_allocated() / 2**30:.2f} GiB")
    return _ORIG_TRAIN(self, *args, **kwargs)


def tally_training_step(self, model, inputs, *args, **kwargs):
    # One very long example should not kill a 40-minute run: on CUDA OOM we
    # skip that micro-batch (logged) and carry on.
    try:
        out = _ORIG_TRAINING_STEP(self, model, inputs, *args, **kwargs)
    except torch.OutOfMemoryError:
        L = int(inputs["input_ids"].shape[-1])
        OOM_SKIPS.append(L)
        print(f"[OOM] skipped one micro-batch of {L} tokens (total skipped: {len(OOM_SKIPS)})")
        gc.collect()
        torch.cuda.empty_cache()
        return torch.tensor(0.0, device=inputs["input_ids"].device)
    if torch.cuda.is_available() and self.state.global_step == 0 and not getattr(self, "_tally_mem_logged", False):
        self._tally_mem_logged = True
        print(f"[mem] peak after first micro-batch: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
    return out


SFTTrainer.compute_loss = tally_compute_loss  # AlignTune's TRL backend uses this same class
SFTTrainer.train = tally_train
SFTTrainer.training_step = tally_training_step
print("trl", trl.__version__, "| answer-only loss, fp32 LoRA cast and OOM guard installed on trl.SFTTrainer")


def history_metrics(hf_trainer):
    # Pull train / eval loss per epoch out of the HF trainer's log history.
    hist = list(getattr(getattr(hf_trainer, "state", None), "log_history", []) or [])
    train_losses = [(h.get("epoch"), h["loss"]) for h in hist if "loss" in h]
    eval_losses = [(h.get("epoch"), h["eval_loss"]) for h in hist if "eval_loss" in h]
    summary = [h for h in hist if "train_runtime" in h]
    return {
        "train_loss_log": train_losses,
        "eval_loss_by_epoch": eval_losses,
        "train_summary": summary[-1] if summary else {},
        "best_model_checkpoint": getattr(getattr(hf_trainer, "state", None), "best_model_checkpoint", None),
    }


def write_sft_dir():
    # AlignTune's DirectoryLoader turns <dir>/train.jsonl + <dir>/validation.jsonl
    # into the train / validation splits (aligntune/data/loaders/directory_loader.py).
    # Passing a bare name like 'tally_sft' makes it try the HF Hub instead.
    d = Path("/kaggle/working/aligntune_data")
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)
    write_jsonl(d / "train.jsonl", [{"messages": r["messages"]} for r in train_rows])
    write_jsonl(d / "validation.jsonl", [{"messages": r["messages"]} for r in val_rows])
    return d


def try_aligntune():
    from aligntune.core.backend_factory import create_sft_trainer

    data_dir = write_sft_dir()
    kwargs = dict(
        model_name=MID,
        dataset_name=str(data_dir),
        backend="trl",
        output_dir=str(RUN_DIR),
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        max_seq_length=MAX_SEQ_LENGTH,
        peft_enabled=True,
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        quantization=QUANTIZATION,
        precision="fp16",
        fp16=True,
        bf16=False,
        gradient_accumulation_steps=GRAD_ACCUM,
        use_gradient_checkpointing=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        assistant_only_loss=False,   # masking done in tally_compute_loss
        task_type="chat_completion",
        curator_schema_gate=True,
        curator_clean=False,
        curator_dedup="none",
        curator_max_tokens=1_000_000,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=1,
        report_to="none",
        seed=SEED,
        warmup_ratio=0.1,
        lr_scheduler="cosine",
        device_map={"": 0},
        compute_rouge=False,
        compute_bleu=False,
        compute_perplexity=False,
        packing=False,
        model_init_kwargs={"cache_dir": HF_CACHE},
        attn_implementation="sdpa",
    )
    print("Creating AlignTune SFT trainer (backend=trl, QLoRA, answer-only loss)...")
    tr = create_sft_trainer(**kwargs)
    print("AlignTune trainer:", type(tr).__name__)
    tr.setup_model()
    print("cast", cast_trainable_fp32(tr.model), "trainable tensors to fp32")
    if hasattr(tr.model, "print_trainable_parameters"):
        tr.model.print_trainable_parameters()
    tr.setup_data()
    tr.dataset = tr.train_dataset
    print("AlignTune data: train", len(tr.train_dataset), "val", len(tr.eval_dataset) if tr.eval_dataset is not None else 0)
    print("Starting AlignTune train()...")
    result = tr.train()
    metrics = {"aligntune_result": {k: v for k, v in (result or {}).items() if k != "metrics"}}
    hf = getattr(tr, "trainer", None)
    if hf is not None:
        try:
            ev = hf.evaluate()
            metrics["final_eval"] = dict(ev)
            print("final eval:", metrics["final_eval"])
        except Exception as exc:
            print("evaluate() skipped:", type(exc).__name__, exc)
        metrics.update(history_metrics(hf))
    return tr, metrics


def try_trl_fallback():
    from peft import LoraConfig
    from trl import SFTConfig

    print("Loading Tiny Aya in 4-bit for the TRL fallback...")
    tok = AutoTokenizer.from_pretrained(MID, cache_dir=HF_CACHE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MID, quantization_config=bnb, device_map={"": 0}, cache_dir=HF_CACHE, dtype=torch.float16,
        attn_implementation="sdpa",
    )
    model.config.use_cache = False
    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    peft_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM", target_modules=LORA_TARGETS,
    )
    raw = dict(
        output_dir=str(RUN_DIR / "trl_fallback"),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        warmup_steps=WARMUP_STEPS,
        logging_steps=1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=MAX_SEQ_LENGTH,
        assistant_only_loss=False,   # masking done in tally_compute_loss
        packing=False,
        report_to="none",
        seed=SEED,
    )
    sig = inspect.signature(SFTConfig.__init__)
    dropped = [k for k in raw if k not in sig.parameters]
    if dropped:
        print("SFTConfig: dropping unsupported args", dropped)
    sft_args = SFTConfig(**{k: v for k, v in raw.items() if k in sig.parameters})

    train_ds = Dataset.from_list([{"messages": r["messages"]} for r in train_rows])
    val_ds = Dataset.from_list([{"messages": r["messages"]} for r in val_rows])
    tr = SFTTrainer(
        model=model, args=sft_args, train_dataset=train_ds, eval_dataset=val_ds,
        peft_config=peft_config, processing_class=tok,
    )
    print("cast", cast_trainable_fp32(tr.model), "trainable tensors to fp32")
    if hasattr(tr.model, "print_trainable_parameters"):
        tr.model.print_trainable_parameters()
    print("TRL SFTTrainer ready. Starting trainer.train()...")
    train_result = tr.train()
    metrics = {"train": dict(getattr(train_result, "metrics", {}) or {})}
    try:
        metrics["final_eval"] = dict(tr.evaluate())
        print("final eval:", metrics["final_eval"])
    except Exception as exc:
        print("evaluate() skipped:", type(exc).__name__, exc)
    metrics.update(history_metrics(tr))
    return tr, metrics


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


try:
    trainer, TRAIN_METRICS = try_aligntune()
    USED_BACKEND = "aligntune-trl"
    print("\nUsed AlignTune (TRL backend).")
except Exception as exc:
    TRAIN_ERRORS["aligntune"] = f"{type(exc).__name__}: {exc}"
    print("\nAlignTune path failed:", TRAIN_ERRORS["aligntune"])
    traceback.print_exc()
    trainer = None

# Run the fallback OUTSIDE the except block: inside it, the exception's traceback
# still references AlignTune's model, so both models would sit on the GPU at once.
if USED_BACKEND is None:
    free_gpu()
    if torch.cuda.is_available():
        print(f"GPU allocated before fallback: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")
    print("Falling back to TRL + PEFT with the same hyperparameters.")
    try:
        trainer, TRAIN_METRICS = try_trl_fallback()
        USED_BACKEND = "trl-peft-fallback"
        print("\nUsed TRL + PEFT fallback.")
    except Exception as exc2:
        TRAIN_ERRORS["trl_fallback"] = f"{type(exc2).__name__}: {exc2}"
        print("\nTRL fallback ALSO failed:", TRAIN_ERRORS["trl_fallback"])
        traceback.print_exc()
        trainer = None
    free_gpu()

TRAIN_METRICS["oom_skipped_microbatches"] = OOM_SKIPS
print("USED_BACKEND =", USED_BACKEND)
print("train/eval loss:", json.dumps({k: TRAIN_METRICS.get(k) for k in ("eval_loss_by_epoch", "final_eval")}, default=str))
(RESULTS_DIR / "train_backend.txt").write_text(f"{USED_BACKEND}\n{json.dumps(TRAIN_ERRORS)}\n", encoding="utf-8")
(RESULTS_DIR / "train_metrics.json").write_text(
    json.dumps({"backend": USED_BACKEND, "errors": TRAIN_ERRORS, "metrics": TRAIN_METRICS}, indent=2, default=str),
    encoding="utf-8",
)


## 5. Save the LoRA adapter

The adapter (the small trained weights, not a full copy of Tiny Aya) is written to `/kaggle/working/tally-lora`. That folder is what you reload later, by loading the base model and then attaching these files. We also try AlignTune's `save_model` when the AlignTune path was used.


In [ ]:
# === 5. SAVE: write the LoRA adapter to /kaggle/working/tally-lora ===
# Never raises: if training failed, the rescoring cells still score the base model.

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)


def adapter_present(d):
    return (d / "adapter_config.json").is_file() and any(d.glob("adapter_model*"))


hf_trainer = getattr(trainer, "trainer", None) or trainer  # AlignTune wraps the HF trainer
if hf_trainer is not None:
    for how in ("save_model", "model.save_pretrained"):
        if adapter_present(ADAPTER_DIR):
            break
        try:
            if how == "save_model":
                hf_trainer.save_model(str(ADAPTER_DIR))
            else:
                hf_trainer.model.save_pretrained(str(ADAPTER_DIR))
            print(how, "->", ADAPTER_DIR)
        except Exception as exc:
            print(how, "failed:", type(exc).__name__, exc)

# Last resort: copy the adapter from the run directory / latest checkpoint.
if not adapter_present(ADAPTER_DIR):
    cands = [p.parent for p in RUN_DIR.rglob("adapter_config.json")]
    cands.sort(key=lambda p: p.stat().st_mtime)
    if cands:
        src = cands[-1]
        print("Copying adapter from", src)
        for item in src.iterdir():
            if item.is_file():
                shutil.copy2(item, ADAPTER_DIR / item.name)

ADAPTER_OK = adapter_present(ADAPTER_DIR)
print("Adapter folder contents:")
for p in sorted(ADAPTER_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(ADAPTER_DIR)}  ({p.stat().st_size:,} bytes)")
print("ADAPTER_OK =", ADAPTER_OK)


## 6. Rescore Miro and Bandai

This is the honest check on the fine-tune: the model never trained on these two projects.

For each validation chunk we send the same user message that is in the curated set, parse the JSON reply, merge the chunk notes with the same `merge_notes` function as the extraction notebook, and score the merged record against the answer key with the same rules as `tinyaya_scores.json` (numeric tolerance, date-range matching, additionality by tool/conclusion family, conflicts counted as unresolved even when gold is one of the candidates).

We free the 4-bit training model, load the base model in float16 with the adapter attached and score it, then unload it and score the plain **base** model on the same chunks. Both runs use float16, so the only difference is the adapter. Comparing both runs on the same windows is fairer than only comparing to the original 38% / 23% figures, which used slightly different chunk boundaries. Those original document-level scores for Miro and Bandai are still printed at the end, as a reference.


In [ ]:
# === 6. HELPERS for generation and scoring ===

import re
from collections import defaultdict

from peft import PeftModel


def parse_json(text):
    # Parse the FIRST complete JSON object. (v4: a reply that repeats the
    # object, e.g. '{...}{...}', used to fail with "Extra data".) Same parser
    # for base and fine-tuned, so the comparison stays fair.
    start = text.find("{")
    if start == -1:
        raise json.JSONDecodeError("no JSON object found", text, 0)
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
        return obj
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.S)
        return json.loads(match.group(0))


def generate_reply(model, tok, prompt, max_new_tokens=900):
    messages = [{"role": "user", "content": prompt}]
    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # stop at <|END_RESPONSE|> or <|END_OF_TURN_TOKEN|>
            eos_token_id=[i for i in (tok.convert_tokens_to_ids("<|END_RESPONSE|>"), tok.eos_token_id) if i is not None],
            pad_token_id=tok.pad_token_id,
        )
    reply_ids = output[0][inputs["input_ids"].shape[1]:]
    return tok.decode(reply_ids, skip_special_tokens=True)


def extract_chunk_text(user_content):
    # Strip the shared extraction prompt, leaving the page-marked chunk.
    marker = "\nPages "
    pos = user_content.find(marker)
    if pos == -1:
        return user_content
    rest = user_content[pos + 1:]  # starts with 'Pages ...'
    # Skip the 'Pages N–M:' header line.
    nl = rest.find("\n")
    return rest[nl + 1:].lstrip() if nl != -1 else rest


def run_val_extraction(model, tok, label):
    # Run every val chunk, merge per document, and score against the key.
    per_doc_notes = defaultdict(list)
    per_doc_text = defaultdict(list)
    raw_rows = []
    for i, (row, meta) in enumerate(zip(val_rows, val_meta)):
        doc = meta["doc"]
        prompt = row["messages"][0]["content"]
        print(f"[{label}] {i + 1}/{len(val_rows)}  {doc} chunk {meta.get('chunk_index')}")
        try:
            reply = generate_reply(model, tok, prompt)
            parsed = parse_json(reply)
            err = None
        except Exception as exc:
            reply = locals().get("reply", "")
            parsed = {"_error": f"{type(exc).__name__}: {exc}", "raw": reply[:2000]}
            err = str(exc)
            print("  parse/generate error:", err)
        per_doc_notes[doc].append(parsed)
        per_doc_text[doc].append(extract_chunk_text(prompt))
        raw_rows.append({
            "doc": doc,
            "chunk_index": meta.get("chunk_index"),
            "reply": reply,
            "parsed": parsed,
            "error": err,
        })
        if torch.cuda.is_available() and (i + 1) % 4 == 0:
            torch.cuda.empty_cache()

    merged_by_doc = {}
    for doc, notes in per_doc_notes.items():
        doc_text = "\n\n".join(per_doc_text[doc])
        merged_by_doc[doc] = merge_notes(notes, doc_text=doc_text)

    scores = score_documents(merged_by_doc, answer_key, docs=sorted(merged_by_doc))
    print()
    print(format_summary_table(scores, f"{label} — Miro + Bandai (same val chunks)"))
    out = {
        "label": label,
        "merged": merged_by_doc,
        "scores": scores,
        "raw": raw_rows,
    }
    path = RESULTS_DIR / f"{label}_val_scores.json"
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print("wrote", path)
    return out


def load_base_model():
    tok = AutoTokenizer.from_pretrained(MID, cache_dir=HF_CACHE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # Inference in float16, matching the extraction notebook, unless memory is tight.
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MID,
            dtype=torch.float16,
            device_map={"": 0},
            cache_dir=HF_CACHE,
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            MID,
            torch_dtype=torch.float16,
            device_map={"": 0},
            cache_dir=HF_CACHE,
        )
    model.eval()
    return tok, model


def release(*objs):
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# === 6a. SCORE the fine-tuned model on the val chunks ===
# Free the 4-bit training model, then load the base model in float16 (same as
# the base run below and the extraction notebook) and attach the saved adapter.
# That way base vs fine-tuned differ only by the adapter.

ft_results = None
try:
    trainer = None
    hf_trainer = None
    free_gpu()
    if not ADAPTER_OK:
        raise RuntimeError("no adapter saved; skipping fine-tuned scoring. Training errors: " + json.dumps(TRAIN_ERRORS))
    ft_tok, base_for_adapter = load_base_model()
    ft_model = PeftModel.from_pretrained(base_for_adapter, str(ADAPTER_DIR))
    ft_model.eval()
    if hasattr(ft_model, "config"):
        ft_model.config.use_cache = True
    ft_results = run_val_extraction(ft_model, ft_tok, "finetuned")
except Exception as exc:
    print("Fine-tuned scoring failed:", type(exc).__name__, exc)
    traceback.print_exc()
finally:
    ft_model = None
    base_for_adapter = None
    free_gpu()


In [ ]:
# === 6b. SCORE the base model on the same val chunks ===

base_results = None
try:
    free_gpu()
    base_tok, base_model = load_base_model()
    base_results = run_val_extraction(base_model, base_tok, "base")
except Exception as exc:
    print("Base scoring failed:", type(exc).__name__, exc)
    traceback.print_exc()
finally:
    base_model = None
    free_gpu()


In [ ]:
# === 6c. COMPARE base vs fine-tuned, and the original extraction scores ===

VAL_DOCS = [
    "2410_Miro-Ghana_Verification_2020-2022",
    "2929_Bandai-Hills-Ghana_Validation_2024",
]

def overall_line(name, score_obj):
    o = score_obj["overall"]
    return (
        f"{name:<28} "
        f"acc+abstention={o['accuracy_incl_abstention']:.1%}  "
        f"stated={o['stated_accuracy']:.1%}  "
        f"correct={o['correct']}  wrong={o['wrong']}  "
        f"conflict={o['conflict-unresolved']}  "
        f"abstained={o['correctly-abstained']}  "
        f"hallucinated={o['hallucinated']}"
    )

def safe(fn, *a):
    try:
        return fn(*a)
    except Exception as exc:
        print("  (could not print:", type(exc).__name__, exc, ")")

orig_val = None
try:
    orig_val = filter_scores_to_docs(base_scores_all, VAL_DOCS)
except Exception as exc:
    print("original scores unavailable:", type(exc).__name__, exc)

bs = base_results["scores"] if base_results else None
fs = ft_results["scores"] if ft_results else None

print("=== Same val chunks ===")
if bs: print(safe(overall_line, "base (this notebook)", bs))
else: print("base: not available")
if fs: print(safe(overall_line, "fine-tuned", fs))
else: print("fine-tuned: not available (see training / scoring errors above)")
if bs: print("\n" + str(safe(format_summary_table, bs, "BASE — this notebook's val chunks")))
if fs: print("\n" + str(safe(format_summary_table, fs, "FINE-TUNED — this notebook's val chunks")))
if orig_val:
    print("\n=== Reference: original extraction notebook scores on these two documents ===")
    print("(Different chunk boundaries, same answer key and scorer.)")
    print(safe(overall_line, "base (original 9-doc run)", orig_val))
    print(safe(format_summary_table, orig_val, "ORIGINAL tinyaya_scores.json — Miro + Bandai only"))

if bs and fs:
    try:
        print("\n=== Field-level stated accuracy, this notebook ===")
        print(f"{'field':<22} {'base stated':>12} {'ft stated':>12} {'base acc+abs':>12} {'ft acc+abs':>12}")
        for f in FIELDS:
            b = bs["per_field"][f]; t = fs["per_field"][f]
            b_s = f"{b['stated_accuracy']:.0%}" if b["stated_accuracy"] is not None else "n/a"
            t_s = f"{t['stated_accuracy']:.0%}" if t["stated_accuracy"] is not None else "n/a"
            print(f"{f:<22} {b_s:>12} {t_s:>12} {b['accuracy_incl_abstention']:>12.0%} {t['accuracy_incl_abstention']:>12.0%}")
        print("\n=== Pairs that changed category (base -> fine-tuned) ===")
        flips = 0
        for a, b in zip(bs["pairs"], fs["pairs"]):
            if a["category"] != b["category"]:
                flips += 1
                print(f"  {a['doc']} / {a['field']}: {a['category']} -> {b['category']}\n"
                      f"      gold={a['gold_value']!r}\n      base={a['extract_value']!r}\n      ft  ={b['extract_value']!r}")
        if flips == 0:
            print("  (no category changes)")
    except Exception as exc:
        print("field-level comparison failed:", type(exc).__name__, exc)

comparison = {
    "backend": USED_BACKEND,
    "train_errors": TRAIN_ERRORS,
    "train_metrics": TRAIN_METRICS,
    "base_this_run": bs,
    "finetuned_this_run": fs,
    "original_miro_bandai": orig_val,
}
(RESULTS_DIR / "comparison.json").write_text(
    json.dumps(comparison, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
# Also put a copy at the top of /kaggle/working so it is easy to download.
(Path("/kaggle/working") / "comparison.json").write_text(
    json.dumps(comparison, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("\nwrote", RESULTS_DIR / "comparison.json")
print("adapter is in", ADAPTER_DIR, "| ADAPTER_OK =", ADAPTER_OK)
print("USED_BACKEND =", USED_BACKEND)


## 7. What to look at, and what this does not yet prove

- **`/kaggle/working/tally-lora`** is the trained adapter. Keep that folder if the session pauses; you can reload it in a later session without training again.
- **`/kaggle/working/tally/finetune_results/comparison.json`** is the base vs fine-tuned score table. The number to quote first is stated accuracy on Miro and Bandai, plus the hallucination count. A small model that admits `"not stated"` can still be the more useful one even if raw accuracy is close.
- Validation loss during training should fall, or at least not rise, across the two epochs. If it rises in epoch 2, the first epoch's checkpoint is the one `load_best_model_at_end` should have kept. Watch the gap between train loss and val loss: a falling train loss with a flat val loss is the overfitting pattern from v4.
- This is **not** the final generalisation test. The held-out documents (5172, 5432, 4329, and North Bandai) stay out of training for that. Additionality now has 21 positive training chunks (8 from v4, 13 new), up from a very thin v4 count.

If the training cell ran out of memory, check that 4-bit quantisation loaded and that the `[answer-only loss]` line shows a small `logits kept` number. Then re-run from the train cell. You do not need to re-run setup unless the session was reset.
